# Tiny LLM Lessons - Colab Runner

Use this notebook to run the tiny LLM project on Google Colab GPU.

Before running: `Runtime -> Change runtime type -> GPU`.

In [ ]:
# Check GPU. If this fails, change runtime type to GPU.
!nvidia-smi

## 1. Install uv

In [ ]:
!pip install -q uv

## 2. Get The Repo

Option A: set `REPO_URL` and clone from GitHub.

Option B: upload the project folder to Colab, then set `PROJECT_DIR` to that folder.

In [ ]:
REPO_URL = ""  # Example: "https://github.com/your-name/your-repo.git"
PROJECT_DIR = "/content/llm"

if REPO_URL:
    !rm -rf "$PROJECT_DIR"
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print("Set REPO_URL above, or upload your project folder and update PROJECT_DIR.")

%cd $PROJECT_DIR

## 3. Prepare Data

In [ ]:
!uv run python download/step12_download_wikitext2.py
!uv run python step15_prepare_wikitext2.py
!uv run python step16_prepare_alpaca.py
!uv run python step19_prepare_assistant_eos.py

## 4. Train EOS BPE Tokenizer

In [ ]:
!uv run python step13_bpe_tokenizer.py \
  --data data/assistant_eos_tokenizer_train.txt \
  --out tokenizers/assistant_eos_bpe_4000.json \
  --vocab-size 4000 \
  --max-chars 3000000 \
  --special-token '<unk>' \
  --special-token '<eos>'

## 5. Pretrain

This trains general next-token prediction on clean WikiText.

In [ ]:
!uv run python step9_train_tiny_transformer.py \
  --preset assistant_eos_pretrain_bpe \
  --device cuda

## 6. SFT

This loads the pretrain checkpoint and trains instruction/response behavior with response-only loss.

In [ ]:
!uv run python step9_train_tiny_transformer.py \
  --preset assistant_eos_sft_bpe \
  --device cuda

## 7. Generate

In [ ]:
!uv run python step10_generate.py \
  --device cuda \
  --prompt "hi, who are you?"

## 8. Save Checkpoints To Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/tiny_llm_checkpoints
!cp -r checkpoints /content/drive/MyDrive/tiny_llm_checkpoints/
!cp -r tokenizers /content/drive/MyDrive/tiny_llm_checkpoints/
!cp README.md PROJECT_PRESENTATION.md COLAB.md /content/drive/MyDrive/tiny_llm_checkpoints/ || true

## Optional: Short Smoke Test

Use this if you only want to test that GPU training starts.

In [ ]:
!uv run python step9_train_tiny_transformer.py \
  --preset assistant_eos_pretrain_bpe \
  --device cuda \
  --max-iters 2 \
  --checkpoint checkpoints/colab_smoke_test.pt